# Model Inference - FER ConvNeXt Pipeline

Loads the trained ConvNeXt-Tiny Facial Expression Recognition pipeline (model weights + preprocessing config) and generates predictions on the Kaggle test set (`icml_face_data.csv`, rows where `Usage == 'PrivateTest'`), writing a `submission.csv` file.

If you logged the pipeline to W&B as an artifact instead of having it locally, download it first using the optional cell below.

###  Download pipeline artifact from W&B



### Authenticate with W&B
To download the model artifact, you need to provide your W&B API key. You can find your key at [wandb.ai/authorize](https://wandb.ai/authorize).

In [3]:
# Install wandb library
!pip install wandb -qqq

# import colab user data
from google.colab import userdata
import wandb

# Log in to wandb non-interactively using the API key from Colab secrets
wandb_api_key = userdata.get('WANDB_API_KEY')
if wandb_api_key:
  print("WANDB_API_KEY found. Attempting non-interactive login to Weights & Biases.")
  wandb.login(key=wandb_api_key)
else:
  print("WANDB_API_KEY not found in environment variables. Attempting interactive login.")
  print("Please ensure your W&B API key is set as a secret named 'WANDB_API_KEY' in Colab or your environment.")
  wandb.login()

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


WANDB_API_KEY found. Attempting non-interactive login to Weights & Biases.


wandb: Currently logged in as: gbera23 (gbera23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [5]:
import wandb
run = wandb.init()
artifact = run.use_artifact("gbera23-free-university-of-tbilisi-/facial-expression-recognition/facial_expression_pipeline:v0", type="model_pipeline")
pipeline_dir = artifact.download()
run.finish()
print(pipeline_dir)

wandb: Downloading large artifact 'facial_expression_pipeline:v0', 106.19MB. 2 files...
wandb:   2 of 2 files downloaded.  
Done. 00:00:02.2 (49.0MB/s)


/content/artifacts/facial_expression_pipeline:v0


### Imports

In [6]:
import json
import os

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import convnext_tiny

### Configuration

In [7]:
DATA_CSV = "./challenges-in-representation-learning-facial-expression-recognition-challenge_data/icml_face_data.csv"
PIPELINE_DIR = "./fer_pipeline_bundle"
USAGE = "PrivateTest"
OUTPUT_CSV = "./submission.csv"
BATCH_SIZE = 64

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


### Model Definition (must match training)

In [8]:
class ConvNeXtFER(nn.Module):
    def __init__(self, num_classes=7, dropout_prob=0.5):
        super(ConvNeXtFER, self).__init__()

        self.backbone = convnext_tiny(weights=None)
        self.backbone.features[0][0] = nn.Conv2d(1, 96, kernel_size=4, stride=4)

        n_inputs = self.backbone.classifier[2].in_features

        self.backbone.classifier = nn.Sequential(
            nn.Flatten(start_dim=1, end_dim=-1),
            nn.LayerNorm((n_inputs,), eps=1e-06, elementwise_affine=True),
            nn.Dropout(p=dropout_prob),
            nn.Linear(n_inputs, num_classes),
        )

    def forward(self, x):
        return self.backbone(x)


def get_model(num_classes=7, dropout_prob=0.5):
    return ConvNeXtFER(num_classes, dropout_prob)

In [19]:
%ls

model.pth  preprocessor_config.json


### Dataset for Inference

In [17]:
class FERTestDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        pixels = self.df.iloc[idx][" pixels"].split()
        image = np.array(pixels, dtype="uint8").reshape(48, 48)

        if self.transform:
            image = self.transform(image)

        return image, idx

### Load Pipeline (model weights + preprocessing config)

In [20]:
def load_pipeline(pipeline_dir, device):
    """Load model weights and preprocessing config from a pipeline bundle dir."""
    config_path = "preprocessor_config.json"
    model_path =  "model.pth"

    with open(config_path, "r") as f:
        config = json.load(f)

    model = get_model(num_classes=config.get("num_classes", 7))
    state_dict = torch.load(model_path, map_location=device)
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()

    # Build the transform from the config
    norm_cfg = config.get("normalization") or config.get("preprocessing", {}).get("normalize", {"mean": [0.5], "std": [0.5]})
    mean = tuple(norm_cfg.get("mean", [0.5]))
    std = tuple(norm_cfg.get("std", [0.5]))

    transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

    emotion_map = config.get("emotion_map", {
        0: "Angry", 1: "Disgust", 2: "Fear", 3: "Happy",
        4: "Sad", 5: "Surprise", 6: "Neutral",
    })
    # JSON keys are strings; convert back to int
    emotion_map = {int(k): v for k, v in emotion_map.items()}

    return model, transform, emotion_map


print(f"Loading pipeline from {PIPELINE_DIR} ...")
model, transform, emotion_map = load_pipeline(PIPELINE_DIR, device)
print("Pipeline loaded.")
print("Emotion map:", emotion_map)

Loading pipeline from ./fer_pipeline_bundle ...
Pipeline loaded.
Emotion map: {0: 'Angry', 1: 'Disgust', 2: 'Fear', 3: 'Happy', 4: 'Sad', 5: 'Surprise', 6: 'Neutral'}


In [39]:
%ls

artifacts/  sample_data/  wandb/


In [40]:
# Install the Kaggle API client
!pip install kaggle -qqq

%cd Machine-Learning/Assignment4

!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


%ls

# Define the competition name
competition_name = 'challenges-in-representation-learning-facial-expression-recognition-challenge'

# Download the competition data
!kaggle competitions download -c {competition_name}
zip_file_name = f"{competition_name}.zip"

!unzip -q {zip_file_name} -d "./{competition_name}_data"

!ls -F "./{competition_name}_data"

print(f"Data unzipped to ./{competition_name}_data")


[Errno 2] No such file or directory: 'Machine-Learning/Assignment4'
/content
artifacts/  sample_data/  wandb/
100% 285M/285M [00:06<00:00, 47.5MB/s]

example_submission.csv	fer2013.tar.gz	icml_face_data.csv  test.csv  train.csv
Data unzipped to ./challenges-in-representation-learning-facial-expression-recognition-challenge_data


### Load Test Data

In [41]:
print(f"Loading data from {DATA_CSV} ...")
df = pd.read_csv(DATA_CSV)
test_df = df[df[" Usage"] == USAGE].reset_index(drop=True)
print(f"Found {len(test_df)} samples with Usage == '{USAGE}'")
test_df.head()

Loading data from ./challenges-in-representation-learning-facial-expression-recognition-challenge_data/icml_face_data.csv ...
Found 3589 samples with Usage == 'PrivateTest'


,emotion,Usage,pixels
0,0,PrivateTest,170 118 101 88 88 75 78 82 66 74 68 59 63 64 6...
1,5,PrivateTest,7 5 8 6 7 3 2 6 5 4 4 5 7 5 5 5 6 7 7 7 10 10 ...
2,6,PrivateTest,232 240 241 239 237 235 246 117 24 24 22 13 12...
3,4,PrivateTest,200 197 149 139 156 89 111 58 62 95 113 117 11...
4,2,PrivateTest,40 28 33 56 45 33 31 78 152 194 200 186 196 20...


### Run Inference

In [42]:
def run_inference(model, transform, df, device, batch_size=64):
    dataset = FERTestDataset(df, transform=transform)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    all_preds = []
    all_indices = []

    with torch.no_grad():
        for images, indices in loader:
            images = images.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            all_preds.extend(preds.cpu().numpy().tolist())
            all_indices.extend(indices.numpy().tolist())

    return all_indices, all_preds


print("Running inference...")
indices, preds = run_inference(model, transform, test_df, device, batch_size=BATCH_SIZE)
print(f"Generated {len(preds)} predictions.")

Running inference...
Generated 3589 predictions.


### Build and Save Submission

In [43]:
submission = pd.DataFrame({
    "id": indices,
    "emotion": preds,
})
submission = submission.sort_values("id").reset_index(drop=True)

submission.to_csv(OUTPUT_CSV, index=False)
print(f"Submission saved to {OUTPUT_CSV}")
submission.head()

Submission saved to ./submission.csv


,id,emotion
0,0,2
1,1,2
2,2,0
3,3,2
4,4,4


### Sanity Check: Predicted Label Distribution

In [44]:
print("Predicted label distribution:")
print(submission["emotion"].map(emotion_map).value_counts())

Predicted label distribution:
emotion
Fear        1590
Neutral      902
Angry        733
Sad          248
Surprise     116
Name: count, dtype: int64
